### AdaptiveRAG


In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["HUGGINGFACEHUB_API_KEY"]=os.getenv("HUGGINGFACEHUB_API_KEY")

In [3]:
#build index
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embd=HuggingFaceEmbeddings()

urls=[
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm",
]

#load
docs=[WebBaseLoader(url).load() for url in urls]
doc_list=[item for sublist in docs for item in sublist]

text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
doc_splits=text_splitter.split_documents(doc_list)

vector_store=FAISS.from_documents(
    documents=doc_splits,
    embedding=HuggingFaceEmbeddings()
)

retriever=vector_store.as_retriever()

In [11]:
#router
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

#data model
class RouteQuery(BaseModel):
    """Route the user query to the most relevant datasource"""
    datasource: Literal["vector_store", "web_search"] = Field(
        description="Given a user question, choose whether to route it to a vector store or web search."
    )

#llm with func call
llm = ChatGroq(model="qwen/qwen3.6-27b")
structured_llm_router = llm.with_structured_output(RouteQuery)

#prompt
system = """You are an expert at routing a user question to a vector store or web search.
The vector store contains documents related to agents, prompt engineering, and adversarial attacks.
Use the vector store for questions on these topics, otherwise, use web search."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

print(
    question_router.invoke(
        {"question": "Who will the bears draft first in the nfl draft?"}
    )
)

datasource='web_search'


In [20]:
#retriever grader
#datamodel
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents"""
    binary_score:str=Field(
        description="Documents are relevant to teh question 'yes' or 'no' ."
    )

#llm with func call
llm = ChatGroq(model="qwen/qwen3.6-27b")
structured_llm_docs = llm.with_structured_output(GradeDocuments)

#prompt
system = """You are a grader assesing relevance of a retrieved document to a user question. \n
    if the document contains keyword(s) or semantic meaning rleated to the user question, grade it as relevant. \n
    it does not need to be a stringent test. the goal is to filter out erroneoud retrievals. \n
    gice a binary score 'yes' or 'no' score to indiicate whether the documnst is relevant to the question."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} User question: {question}"),
    ]
)

retrieval_grader=grade_prompt|structured_llm_docs
question="agent memory"
docs=retriever.invoke(question)
doc_txt=docs[1].page_content
print(retrieval_grader.invoke({"question":question,"document":doc_txt}))

binary_score='yes'


In [19]:
#Generate 
from langchain_classic import hub
from langsmith import Client
from langchain_core.output_parsers import StrOutputParser
client=Client()

#prompt
# prompt = hub.pull("rlm/rag-prompt", dangerously_pull_public_prompt=True)
prompt =client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

#llm
llm = ChatGroq(model="qwen/qwen3.6-27b")

#post-preprocessing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#chain
rag_chain=prompt|llm|StrOutputParser()

#run
generation=rag_chain.invoke({"context":docs,"question":question})
print(generation)
# rag_chain = (
#     {"context": format_docs, "question": RunnablePassthrough()}
#     | prompt
#     | llm
#     | StrOutputParser()
# )

# generation = rag_chain.invoke(question)



<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** agent memory
   - **Context:** Multiple excerpts from Lilian Weng's blog post "LLM Powered Autonomous Agents". It explicitly defines two types of agent memory: Short-term memory (in-context learning) and Long-term memory (retaining/recalling information over extended periods using external vector stores and fast retrieval).
   - **Constraints:** Use retrieved context, say "I don't know" if unsure, max 3 sentences, keep it concise.

2.  **Extract Key Information from Context:**
   - Agent memory in LLM-powered systems consists of two main types: short-term and long-term.
   - Short-term memory refers to in-context learning, where the model learns from the immediate prompt/context.
   - Long-term memory enables the agent to retain and recall information over extended periods, typically using external vector stores and fast retrieval mechanisms.

3.  **Draft Response (mental refinement):**
   In LLM-powere

In [21]:
#hallucination grader

#data model
class GradeHallucination(BaseModel):
    """Binary score for hallucination present in generation answer."""
    binary_score:str=Field(description="Answer is grounded in, 'yes' or 'no' .")


#llm with func call
llm = ChatGroq(model="qwen/qwen3.6-27b")
structured_llm_grader = llm.with_structured_output(GradeDocuments)

#prompt
system = """You are a grader assesing whether an llm generation is grounded in / supported by a set of retrieved facts. \n
    give a binary score 'yes' or 'no', 'Yes' means that the nswer is grounded in /supportedby the set of facts."""

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "set of facts: /n/n {documents} \n\n LLM generation: {generation}"),
    ]
)
hallucination_grader=hallucination_prompt|structured_llm_grader
hallucination_grader.invoke({"documents":docs,"generation":generation})


GradeDocuments(binary_score='yes')

In [23]:
#answer grader

#data model
class GradeAnswer(BaseModel):
    """Binary score on whether answer addresses question."""
    binary_score:str=Field(description="Answer addresses the question, 'yes' or 'no' .")


#llm with func call
llm = ChatGroq(model="qwen/qwen3.6-27b")
structured_llm_answer = llm.with_structured_output(GradeAnswer)

#prompt
system = """You are a grader assessing whether an llm generation addresses the user question. \n
    if the answer directly addresses the user question and provides relevant information, grade it as 'yes'. \n
    if the answer does not address the question or is off-topic, grade it as 'no'. \n
    give a binary score 'yes' or 'no' to indicate whether the answer addresses the question."""

answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structured_llm_answer
answer_grader.invoke({"question":question,"generation":generation})

GradeAnswer(binary_score='yes')

In [24]:
#rewrite question

#llm
llm = ChatGroq(model="qwen/qwen3.6-27b")

#prompt
system = """You are a question re-writer. \n
    Given a user question, your task is to reformulate it into a better question for retrieval. \n
    the re-written question should be more specific, clear, and optimized for semantic search. \n
    ensure the re-written question captures the core intent of the original question."""

re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Original question: {question}"),
    ]
)

question_rewriter = re_write_prompt|llm| StrOutputParser()
question_rewriter.invoke({"question": question})

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Original question: "agent memory"\n   - Intent: The user is likely asking about how memory works in AI agents, what types of agent memory exist, how to implement it, or its importance in AI systems.\n   - Characteristics: Very short, vague, lacks context, not optimized for semantic search.\n\n2.  **Identify Core Concepts:**\n   - AI agents\n   - Memory mechanisms/systems\n   - Types of memory (short-term, long-term, working, episodic, semantic, etc.)\n   - Implementation/architecture\n   - Purpose/function (context retention, learning, decision-making)\n\n3.  **Determine Goal for Rewriting:**\n   - Make it specific and clear\n   - Optimize for semantic search/retrieval\n   - Capture the core intent (understanding/implementing memory in AI agents)\n   - Provide a well-structured question that would yield relevant technical/conceptual results\n\n4.  **Brainstorming Rewrites:**\n   - "How does memory work in AI a

In [26]:
#graph state and workflow
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.types import Send

# Define GraphState
class GraphState(TypedDict):
    question: str
    generation: str
    documents: list
    web_search: str
    route: str

# Node functions
def route_query(state):
    """Route query to vector store or web search"""
    question = state["question"]
    router_result = question_router.invoke({"question": question})
    route = router_result.datasource
    return {"route": route}

def retrieve(state):
    """Retrieve documents from vector store"""
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents}

def grade_documents(state):
    """Grade relevance of retrieved documents"""
    question = state["question"]
    documents = state["documents"]
    
    filtered_docs = []
    for doc in documents:
        grade = retrieval_grader.invoke({"question": question, "document": doc.page_content})
        if grade.binary_score == "yes":
            filtered_docs.append(doc)
    
    return {"documents": filtered_docs}

def generate(state):
    """Generate answer from retrieved documents"""
    question = state["question"]
    documents = state["documents"]
    
    generation = rag_chain.invoke({"context": documents, "question": question})
    return {"generation": generation}

def grade_hallucination(state):
    """Check if generation is grounded in documents"""
    generation = state["generation"]
    documents = state["documents"]
    
    hallucination_check = hallucination_grader.invoke({"documents": documents, "generation": generation})
    return {"generation": generation if hallucination_check.binary_score == "yes" else "Hallucination detected"}

def grade_answer(state):
    """Check if answer addresses the question"""
    question = state["question"]
    generation = state["generation"]
    
    answer_check = answer_grader.invoke({"question": question, "generation": generation})
    return {"generation": generation if answer_check.binary_score == "yes" else "Answer does not address question"}

def rewrite_question(state):
    """Rewrite question for better retrieval"""
    question = state["question"]
    new_question = question_rewriter.invoke({"question": question})
    return {"question": new_question}

def web_search(state):
    """Perform web search"""
    question = state["question"]
    # Placeholder for web search
    web_result = f"Web search results for: {question}"
    return {"web_search": web_result}

# Build the graph
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("route_query", route_query)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("grade_hallucination", grade_hallucination)
workflow.add_node("grade_answer", grade_answer)
workflow.add_node("rewrite_question", rewrite_question)
workflow.add_node("web_search", web_search)

# Set entry point
workflow.set_entry_point("route_query")

# Add edges
workflow.add_edge("route_query", "retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_edge("grade_documents", "generate")
workflow.add_edge("generate", "grade_hallucination")
workflow.add_edge("grade_hallucination", "grade_answer")
workflow.add_edge("grade_answer", END)
workflow.add_edge("web_search", END)

# Compile graph
app = workflow.compile()

# # Visualize the graph
# try:
#     graph_image = app.get_graph().draw_mermaid_png()
#     from IPython.display import Image, display
#     display(Image(data=graph_image))
# except Exception as e:
#     print(f"Visualization error: {e}")
#     print(app.get_graph().draw_mermaid())

# # Run the workflow
# print("\n" + "="*80)
# print("ADAPTIVE RAG WORKFLOW EXECUTION")
# print("="*80)

# initial_state = {
#     "question": question,
#     "generation": "",
#     "documents": [],
#     "web_search": "",
#     "route": ""
# }

# # Execute
# result = app.invoke(initial_state)

# print("\n" + "="*80)
# print("FINAL RESULTS")
# print("="*80)
# print(f"\nQuestion: {result['question']}")
# print(f"\nRoute: {result['route']}")
# print(f"\nDocuments Retrieved: {len(result['documents'])}")
# print(f"\nGeneration:\n{result['generation']}")
# print("\n" + "="*80)

In [29]:
# Test full graph execution with error handling
print("="*80)
print("Testing Full Graph Execution")
print("="*80)

try:
    initial_state = {
        "question": "agent memory",
        "generation": "",
        "documents": [],
        "web_search": "",
        "route": ""
    }
    
    result = app.invoke(initial_state)
    
    print("\n" + "="*80)
    print("SUCCESS - FINAL RESULTS")
    print("="*80)
    print(f"\nQuestion: {result['question']}")
    print(f"\nRoute: {result.get('route', 'N/A')}")
    print(f"\nDocuments Retrieved: {len(result['documents'])}")
    if result['generation']:
        print(f"\nGeneration:\n{result['generation'][:500]}...")
    print("\n" + "="*80)
    
except Exception as e:
    print(f"\nError: {type(e).__name__}")
    print(f"Message: {str(e)[:500]}")
    import traceback
    print("\nTraceback:")
    traceback.print_exc()

# Try streaming for better visibility
print("\n" + "="*80)
print("Streaming Execution for Debugging")
print("="*80)

try:
    for step in app.stream(initial_state):
        print(f"\nStep: {step}")
except Exception as e:
    print(f"Stream error: {e}")

Testing Full Graph Execution

SUCCESS - FINAL RESULTS

Question: agent memory

Route: vector_store

Documents Retrieved: 4

Generation:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "agent memory"
   - **Context:** Multiple excerpts from Lilian Weng's blog post "LLM Powered Autonomous Agents". The context repeatedly mentions the "Memory" component of LLM-powered autonomous agents.
   - **Key Information in Context about Memory:**
     - Short-term memory: Utilizes in-context learning to learn.
     - Long-term memory: Provides capability to retain and recall (infinite) information over exten...


Streaming Execution for Debugging

Step: {'route_query': {'route': 'vector_store'}}

Step: {'retrieve': {'documents': [Document(id='9ebb118c-518a-429e-81ae-f66fd54b155c', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language mo